# Exercise 4.1 — Unified modelling frameworks in R and Python

*HDS MSc · Module 3 · the Python version of this exercise; the web page is at <https://inflamuob.github.io/HDS_M3/day4/day4-01-frameworks.html>.*

*Open it in Jupyter or VS Code and run it a cell at a time. Keep it in the folder that holds your `data` folder. Cells with `______` are blanks for you to fill in.*

In both R and Python there are many ways to fit a model:

- **R:** `lm()`, `glm()`, `rpart()`, `randomForest()`, `xgboost()`, …
- **Python:** `statsmodels`, your own NumPy code, or one of the many classes in
  scikit-learn.

Each function or class tends to have its own syntax, its own argument names and
its own way of returning a prediction. That becomes a problem as soon as we
want to:

- swap one model for another,
- compare models fairly,
- build a whole pipeline of preprocessing, model and evaluation,
- tune a model without leaking information from the test data.

Two ecosystems solve this: **tidymodels** in R and **scikit-learn** in Python.
Both put one consistent interface in front of the models, the preprocessing,
the pipelines and the tuning.

| The question | tidymodels (R) | scikit-learn (Python) |
|---|---|---|
| what kind of model is this? | `logistic_reg()`, `decision_tree()` | the estimator class, `LogisticRegression` |
| which code actually fits it? | `set_engine("glm")` | the class, plus `solver=` |
| what happens to the data first? | a `recipe()` of `step_*()` | transformers, e.g. `StandardScaler` |
| the two joined together | `workflow()` | `Pipeline` |
| fit it, then use it | `fit()`, `predict()` | `.fit()`, `.predict()` |

**Theory:** model specification and engine; the fit/predict interface;
predicted probabilities; pipelines that carry their preprocessing with them.

Useful documentation: [Getting started with tidymodels](https://www.tidymodels.org/start/),
the [parsnip list of models](https://parsnip.tidymodels.org/reference/index.html),
and the [scikit-learn user guide](https://scikit-learn.org/stable/user_guide.html).

> **The plots are for looking at**
>
> The code for every plot is included, so that you can run it and see the figure. You are not expected to write it yourself.

### Packages

Everything in this exercise comes from scikit-learn, imported one piece at a
time as we need it.

In [ ]:
# Packages for this exercise. If one is missing, install it once from a terminal
# (in a Jupyter notebook, type %pip instead of pip):
# pip install pandas numpy matplotlib scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

------------------------------------------------------------------------

### Part 1 — What model, and which engine

In scikit-learn every model is a class, called an **estimator**, and every
estimator is used in the same way:

```python
model = SomeEstimator(settings)
model = model.fit(X_train, y_train)
predictions = model.predict(X_test)
```

Creating the estimator declares the model. Here are four classifiers:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

log_spec  = LogisticRegression()
tree_spec = DecisionTreeClassifier()
rf_spec   = RandomForestClassifier()
knn_spec  = KNeighborsClassifier()

print(log_spec)

In [ ]:
print(tree_spec)

This does not fit anything yet. It only says "I want a logistic regression
classifier", "I want a decision tree", and so on. There is no data in it, and no
coefficients.

scikit-learn has no separate engine. The closest thing is the `solver=`
argument, which picks the numerical method used to fit the model, the same idea
as an engine in tidymodels:

In [ ]:
log_spec_lbfgs     = LogisticRegression(solver="lbfgs")       # the default
log_spec_liblinear = LogisticRegression(solver="liblinear")   # another algorithm

print(log_spec_liblinear)

> **Learning points**
>
> - A **specification** says what kind of model you want; it has not seen any data.
> - The **engine** (in Python, the estimator and its `solver`) is the code that fits it.
> - Every model is then used with the same two verbs: fit it, then predict with it.

------------------------------------------------------------------------

### Part 2 — Fit and predict a logistic regression

**The data.** `data/diabetes_clean.csv` holds 750 adults screened for type 2
diabetes: eight measurements from a single visit, and `diabetes`, which is
`pos` or `neg`. It is the Day 2 file with the impossible readings of 0 already
taken out, so it is ready to use. We keep glucose, BMI (`mass`), age and the
outcome.

**The split.** We hold back every fifth person as the test set: 600 people to
fit the models on, and 150 to test them. Splitting by a fixed rule rather than
at random gives both languages exactly the same 150 people, so the R and the
Python tabs print the same numbers. Every Day 4 exercise uses this split.

In [ ]:
diabetes = pd.read_csv("data/diabetes_clean.csv")
diabetes = diabetes[["glucose", "mass", "age", "diabetes"]]
diabetes["diabetes"] = (diabetes["diabetes"] == "pos").astype(int)       # 1 = pos, 0 = neg

is_test = diabetes.index % 5 == 0               # every fifth person is held back
train = diabetes[~is_test]
test  = diabetes[is_test]

train["diabetes"].value_counts()

scikit-learn wants the outcome as numbers, so `pos` became 1 and `neg` became
0. The model then predicts the probability of 1: having diabetes.

Now fit the model, predicting diabetes from glucose only. `C=np.inf` switches
off a penalty that scikit-learn adds to logistic regression by default; without
it the coefficients would not match the ones R reports.

In [ ]:
X_train = train[["glucose"]]     # two-dimensional: a data frame with one column
y_train = train["diabetes"]      # one-dimensional: a single column

logistic_fit = LogisticRegression(C=np.inf)
logistic_fit = logistic_fit.fit(X_train, y_train)

print(logistic_fit.intercept_, logistic_fit.coef_)

These are the same two coefficients R reports. They are on the **log-odds**
scale, which Exercise 4.2 explains.

**Predictions.** Every scikit-learn classifier has the same `.predict_proba()`.
It returns one column per class, in the order of `classes_`:

In [ ]:
print(logistic_fit.classes_)    # column 0 is class 0 (neg), column 1 is class 1 (pos)

probs = logistic_fit.predict_proba(test[["glucose"]])

test_preds = test.copy()
test_preds["prob_neg"] = probs[:, 0]
test_preds["prob_pos"] = probs[:, 1]

test_preds.head()

`prob_pos` is the predicted probability of diabetes, $\hat{P}(\text{pos})$, and
`prob_neg` is $\hat{P}(\text{neg}) = 1 - \hat{P}(\text{pos})$. Plot the
predicted probability against glucose:

In [ ]:
fig, ax = plt.subplots()
ax.scatter(test_preds["glucose"], test_preds["prob_pos"], alpha=0.5)
ax.set_xlabel("Glucose (mg/dL)")
ax.set_ylabel("Predicted probability of diabetes")
plt.show()

The points lie on an S-shaped curve: the higher the glucose, the higher the
predicted probability, but never below 0 or above 1.

> **Learning points**
>
> - One interface: the same `fit()` and `predict()` work for every model in the framework.
> - A classifier can give a predicted **probability** for each class, not only a class.
> - Check which class the probability is for: in R the second factor level, in Python the column of `classes_`.

------------------------------------------------------------------------

### Part 3 — Pipelines

scikit-learn chains the steps before the model and the model itself in a
**Pipeline**. A preprocessing step, such as `StandardScaler`, is called a
*transformer*.

Here the pipeline standardises the three predictors, as on Day 2, and then fits
the logistic regression:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

features = ["glucose", "mass", "age"]

log_pipe = Pipeline([
    ("scale", StandardScaler()),                 # preprocessing step
    ("model", LogisticRegression(C=np.inf)),     # the model
])

log_pipe = log_pipe.fit(train[features], train["diabetes"])

print(log_pipe)

The pipeline holds the preprocessing as well as the model. That is the point:
the standardisation was learned from the training rows, and `.predict()` applies
exactly the same standardisation to any new rows before the model sees them.

How often is the pipeline right on the 150 test patients? `.score()` gives the
accuracy, the proportion of correct predicted classes, using a cut-off of 0.5.
Compare it with a "model" that always says `neg` (0), the most common class:

In [ ]:
print("accuracy:       ", round(log_pipe.score(test[features], test["diabetes"]), 3))
print("always says neg:", round((test["diabetes"] == 0).mean(), 3))

The logistic regression is right for 74.0% of the test patients. Always saying
`neg` is right for 64.7%, because 97 of the 150 do not have diabetes. Three
clinical measurements add about nine percentage points over a rule that looks
at nothing.

**Swap the model, keep everything else.** This is what the frameworks are for.
Replace the logistic regression with a decision tree and change nothing else:
the same recipe, the same rows, the same accuracy calculation.

In [ ]:
# Replace every ______ below, then run this cell.
from sklearn.tree import DecisionTreeClassifier

tree_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("model", ______(random_state=1)),
])

tree_pipe = tree_pipe.fit(train[features], train["diabetes"])

print("accuracy:", round(tree_pipe.score(______, test["diabetes"]), 3))

> **Stuck?**
>
> The estimator is the tree class you imported. The pipeline is scored on the
> test patients' three predictors, exactly as the logistic regression was.

**Question!** The two trees are given the same 600 people and the same three
predictors, and the word "decision tree" is the same in both languages. Why do
they disagree by more than ten percentage points?

The defaults differ. `rpart` stops growing the tree when a further split no
longer pays for itself, and ends with 6 leaves. `DecisionTreeClassifier` keeps
splitting until every leaf holds only one class, and ends with 171 leaves: it
has learned the training patients almost one by one, and does worse on new
ones. Same name, same data, different defaults, and the defaults are the whole
difference. Always check what a model's defaults are before you compare it with
anything.

> **Learning points**
>
> - A recipe (R) or the steps of a Pipeline (Python) hold the preprocessing; a workflow or Pipeline joins it to the model.
> - The preprocessing is learned from the training rows only and is reapplied, unchanged, to new rows.
> - Swapping the model is one line; everything else stays the same, so the comparison is fair.
> - Compare every model with a baseline that does no learning, such as always predicting the most common class.

## Stretch

> **Stretch — how much of the accuracy is the split?**
>
> Change the first row of the test set: hold back every fifth person starting
> from row 2, then from row 3, 4 and 5, and refit the logistic regression
> workflow each time. Write down the five accuracies. How far apart are they, and
> what does that say about quoting an accuracy from a single split of 150 people?